# CHANCE-C Configuration Deep Dive

Welcome to the comprehensive guide on CHANCE-C configuration management! This notebook will teach you everything you need to know about:

- Understanding YAML configuration files
- Creating and modifying configurations
- Accessing configuration from Model instances
- Saving and loading configurations
- Best practices for configuration management

Let's dive in!


## 1. Introduction to CHANCE-C Configuration

CHANCE-C uses a comprehensive configuration system built around the `SimulationConfig` class and YAML files. This system allows you to:

- **Define all simulation parameters** in one place
- **Version control your experiments** by saving configuration files
- **Reproduce results** by loading exact configurations
- **Share configurations** with collaborators
- **Programmatically modify** parameters for sensitivity analysis

### Configuration Architecture

The configuration system has three main components:

1. **SimulationConfig Class**: A Python dataclass that holds all parameters
2. **YAML Files**: Human-readable configuration files
3. **Model Integration**: Seamless access from the Model class


In [1]:
# Import necessary modules
import os
import yaml
from pathlib import Path

# Import CHANCE-C components
from chance_c import Model, SimulationConfig
from chance_c.field_mapper import FieldMapper
from chance_c.data_loader import get_example_config_path, get_example_field_mapping_path


## 2. Understanding the SimulationConfig Class

The `SimulationConfig` class is a Python dataclass that contains all the parameters needed to run a CHANCE-C simulation. Let's explore its structure and capabilities.


In [2]:
# Create a default configuration
config = SimulationConfig()

print("=== SimulationConfig Overview ===")
print(f"Configuration class: {type(config).__name__}")
print(f"Number of parameters: {len(config.__dict__)}")
print()

# Display key parameter categories
print("=== Parameter Categories ===")
print("Simulation Setup:")
print(f"  - simulation_name: {config.simulation_name}")
print(f"  - scenario: {config.scenario}")
print(f"  - start_year: {config.start_year}")
print(f"  - n_years: {config.n_years}")
print()

print("Agent Parameters:")
print(f"  - agent_housing_aggregation: {config.agent_housing_aggregation}")
print(f"  - household_size: {config.household_size}")
print(f"  - initial_vacancy: {config.initial_vacancy}")
print()

print("Growth Parameters:")
print(f"  - pop_growth_perc: {config.pop_growth_perc}")
print(f"  - inc_growth_perc: {config.inc_growth_perc}")
print(f"  - perc_move: {config.perc_move}")
print()

print("Data Files:")
print(f"  - geo_filename: {os.path.basename(config.geo_filename)}")
print(f"  - pop_filename: {os.path.basename(config.pop_filename)}")
print()

print("Field Mappings:")
print(f"  - geo_file_mapping: {config.geo_file_mapping is not None}")
print(f"  - pop_file_mapping: {config.pop_file_mapping is not None}")
print(f"  - flood_file_mapping: {config.flood_file_mapping is not None}")
print(f"  - housing_file_mapping: {config.housing_file_mapping is not None}")
print(f"  - hedonic_file_mapping: {config.hedonic_file_mapping is not None}")


=== SimulationConfig Overview ===
Configuration class: SimulationConfig
Number of parameters: 40

=== Parameter Categories ===
Simulation Setup:
  - simulation_name: ABM_Baltimore_example
  - scenario: Baseline
  - start_year: 2018
  - n_years: 2

Agent Parameters:
  - agent_housing_aggregation: 10
  - household_size: 2.7
  - initial_vacancy: 0.2

Growth Parameters:
  - pop_growth_perc: 0.01
  - inc_growth_perc: 0.05
  - perc_move: 0.1

Data Files:
  - geo_filename: block_group_extract.shp
  - pop_filename: block_group_population_2018.csv

Field Mappings:
  - geo_file_mapping: True
  - pop_file_mapping: True
  - flood_file_mapping: True
  - housing_file_mapping: True
  - hedonic_file_mapping: True


### 2.1 Complete Parameter List

Let's examine all available parameters in the SimulationConfig class:


In [3]:
# Display all configuration parameters with their values and types
print("=== Complete Configuration Parameters ===")
print()

for param_name, param_value in config.__dict__.items():
    param_type = type(param_value).__name__
    if isinstance(param_value, str) and len(param_value) > 50:
        display_value = param_value[:47] + "..."
    else:
        display_value = param_value
    
    print(f"{param_name:25} ({param_type:8}): {display_value}")

print(f"\nTotal parameters: {len(config.__dict__)}")


=== Complete Configuration Parameters ===

simulation_name           (str     ): ABM_Baltimore_example
scenario                  (str     ): Baseline
intervention              (str     ): Baseline
start_year                (int     ): 2018
n_years                   (int     ): 2
agent_housing_aggregation (int     ): 10
household_size            (float   ): 2.7
initial_vacancy           (float   ): 0.2
pop_growth_mode           (str     ): perc
pop_growth_perc           (float   ): 0.01
inc_growth_mode           (str     ): random_agent_replication
pop_growth_inc_perc       (float   ): 0.9
inc_growth_perc           (float   ): 0.05
bld_growth_perc           (float   ): 0.01
perc_move                 (float   ): 0.1
perc_move_mode            (str     ): random
house_budget_mode         (str     ): rhea
house_choice_mode         (str     ): simple_avoidance_utility
simple_anova_coefficients (tuple   ): (-121428, 294707, 130553, 128990, 154887, -500000)
simple_avoidance_perc     (float   )

## 3. Working with YAML Configuration Files

YAML (Yet Another Markup Language) is a human-readable data serialization standard. CHANCE-C uses YAML files to store and load configuration parameters.

### 3.1 Examining the Example Configuration


In [4]:
# Load and display the example configuration file
example_config_path = get_example_config_path()

print("=== Example Configuration File ===")
print(f"File path: {example_config_path}")
print()

# Read and display the YAML content
if os.path.exists(example_config_path):
    with open(example_config_path, 'r') as file:
        yaml_content = file.read()
    
    print("YAML Content (first 70 lines):")
    print("-" * 50)
    lines = yaml_content.split('\n')
    for i, line in enumerate(lines[:70], 1):
        print(f"{i:2d}: {line}")
    
    if len(lines) > 70:
        print(f"... ({len(lines) - 70} more lines)")
    
    # Highlight the new inline field mapping sections
    print()
    print("=== Key Features of New Configuration Format ===")
    print("All settings consolidated in one file")
    print("Inline field mappings for each data type")
    print("Comprehensive comments explaining each field")
    print("Easy customization for different column names")
    print("No separate field mapping files needed")
else:
    print("Example configuration file not found!")


=== Example Configuration File ===
File path: /Users/d3y010/repos/github/icom_abm/chance_c/data/example_config.yml

YAML Content (first 70 lines):
--------------------------------------------------
 1: # Example configuration file for CHANCE ABM simulation
 2: # This file shows how to configure the simulation with field mapping
 3: 
 4: # Simulation setup
 5: simulation_name: "ABM_Baltimore_example"
 6: scenario: "Baseline"
 7: intervention: "Baseline"
 8: start_year: 2018
 9: n_years: 2
10: 
11: # Agent and housing parameters
12: agent_housing_aggregation: 10
13: household_size: 2.7
14: initial_vacancy: 0.20
15: 
16: # Population growth parameters
17: pop_growth_mode: "perc"
18: pop_growth_perc: 0.01
19: 
20: # Income growth parameters
21: inc_growth_mode: "random_agent_replication"
22: pop_growth_inc_perc: 0.90
23: inc_growth_perc: 0.05
24: 
25: # Building and movement parameters
26: bld_growth_perc: 0.01
27: perc_move: 0.10
28: perc_move_mode: "random"
29: 
30: # Housing choice para

### 3.2 Loading Configuration from YAML

Let's load a configuration from the YAML file and compare it with the default configuration:


In [5]:
# Load configuration from YAML file
if os.path.exists(example_config_path):
    yaml_config = SimulationConfig.from_yaml(example_config_path)
    
    print("=== Configuration Loaded from YAML ===")
    print(f"Simulation name: {yaml_config.simulation_name}")
    print(f"Scenario: {yaml_config.scenario}")
    print(f"Start year: {yaml_config.start_year}")
    print(f"Number of years: {yaml_config.n_years}")
    print()
    
    # Show the field mappings
    print("=== Field Mappings Loaded ===")
    print(f"Geographic file mapping: {len(yaml_config.geo_file_mapping)} fields")
    print(f"Population file mapping: {len(yaml_config.pop_file_mapping)} fields")
    print(f"Flood file mapping: {len(yaml_config.flood_file_mapping)} fields")
    print(f"Housing file mapping: {len(yaml_config.housing_file_mapping)} fields")
    print(f"Hedonic file mapping: {len(yaml_config.hedonic_file_mapping)} fields")
    print()
    
    # Show example field mappings
    print("=== Example Field Mappings ===")
    print("Geographic file mappings:")
    for field, mapping in list(yaml_config.geo_file_mapping.items())[:3]:
        print(f"  {field} → {mapping}")
    print()
    
    print("Population file mappings:")
    for field, mapping in yaml_config.pop_file_mapping.items():
        print(f"  {field} → {mapping}")
    print()
    
    # Compare with default configuration
    print("=== Differences from Default Configuration ===")
    default_config = SimulationConfig()
    
    differences = []
    for param_name in yaml_config.__dict__:
        yaml_value = getattr(yaml_config, param_name)
        default_value = getattr(default_config, param_name)
        
        if yaml_value != default_value:
            differences.append((param_name, default_value, yaml_value))
    
    if differences:
        print(f"Found {len(differences)} differences:")
        # Show only non-mapping differences for clarity
        for param_name, default_val, yaml_val in differences:
            if not param_name.endswith('_mapping'):
                print(f"  {param_name}:")
                print(f"    Default: {default_val}")
                print(f"    YAML:    {yaml_val}")
                print()
        print(f"Note: Field mappings are identical to defaults in this example")
    else:
        print("No differences found - YAML matches defaults")
else:
    print("Cannot load YAML configuration - file not found")


=== Configuration Loaded from YAML ===
Simulation name: ABM_Baltimore_example
Scenario: Baseline
Start year: 2018
Number of years: 2

=== Field Mappings Loaded ===
Geographic file mapping: 7 fields
Population file mapping: 2 fields
Flood file mapping: 4 fields
Housing file mapping: 9 fields
Hedonic file mapping: 7 fields

=== Example Field Mappings ===
Geographic file mappings:
  GISJOIN → GISJOIN
  GEOID → GEOID
  COUNTYFP → COUNTYFP

Population file mappings:
  GISJOIN → GISJOIN
  AJWME001 → AJWME001

=== Differences from Default Configuration ===
Found 1 differences:
  simple_anova_coefficients:
    Default: (-121428, 294707, 130553, 128990, 154887, -500000)
    YAML:    [-121428, 294707, 130553, 128990, 154887, -500000]

Note: Field mappings are identical to defaults in this example


In [6]:
# Demonstrate creating custom field mappings
print("=== Creating Custom Field Mappings ===")

# Example: Your data files use different column names
custom_config_with_mappings = SimulationConfig(
    simulation_name="Custom_Field_Mapping_Example",
    scenario="Different_Column_Names",
    
    # Custom geographic file mappings
    geo_file_mapping={
        "GISJOIN": "BLOCK_GROUP_ID",      # Your file uses BLOCK_GROUP_ID
        "GEOID": "CENSUS_ID",             # Your file uses CENSUS_ID  
        "COUNTYFP": "COUNTY_CODE",        # Your file uses COUNTY_CODE
        "TRACTCE": "TRACT_CODE",          # Your file uses TRACT_CODE
        "BLKGRPCE": "BLOCK_GROUP_CODE",   # Your file uses BLOCK_GROUP_CODE
        "ALAND": "LAND_AREA",             # Your file uses LAND_AREA
        "geometry": "geom"                # Your file uses geom
    },
    
    # Custom population file mappings
    pop_file_mapping={
        "GISJOIN": "BLOCK_GROUP_ID",      # Match the geo file
        "AJWME001": "TOTAL_POPULATION"    # Your file uses TOTAL_POPULATION
    },
    
    # Custom flood file mappings
    flood_file_mapping={
        "GISJOIN": "BLOCK_GROUP_ID",      # Match the geo file
        "Shape_Area": "TOTAL_AREA",       # Your file uses TOTAL_AREA
        "fld_area": "FLOOD_AREA",         # Your file uses FLOOD_AREA
        "perc_fld_area": "FLOOD_PERCENT"  # Your file uses FLOOD_PERCENT
    }
)

print("Custom field mappings created!")
print()

print("Geographic file mappings:")
for required_field, your_column in custom_config_with_mappings.geo_file_mapping.items():
    print(f"  {required_field:12} → {your_column}")
print()

print("Population file mappings:")
for required_field, your_column in custom_config_with_mappings.pop_file_mapping.items():
    print(f"  {required_field:12} → {your_column}")
print()

print("Flood file mappings:")
for required_field, your_column in custom_config_with_mappings.flood_file_mapping.items():
    print(f"  {required_field:15} → {your_column}")

print()
print("=== Benefits of Inline Field Mappings ===")
print("No separate field mapping files to manage")
print("All configuration in one place")
print("Easy to version control and share")
print("Inline documentation with comments")
print("Reduced chance of file path errors")


=== Creating Custom Field Mappings ===
Custom field mappings created!

Geographic file mappings:
  GISJOIN      → BLOCK_GROUP_ID
  GEOID        → CENSUS_ID
  COUNTYFP     → COUNTY_CODE
  TRACTCE      → TRACT_CODE
  BLKGRPCE     → BLOCK_GROUP_CODE
  ALAND        → LAND_AREA
  geometry     → geom

Population file mappings:
  GISJOIN      → BLOCK_GROUP_ID
  AJWME001     → TOTAL_POPULATION

Flood file mappings:
  GISJOIN         → BLOCK_GROUP_ID
  Shape_Area      → TOTAL_AREA
  fld_area        → FLOOD_AREA
  perc_fld_area   → FLOOD_PERCENT

=== Benefits of Inline Field Mappings ===
No separate field mapping files to manage
All configuration in one place
Easy to version control and share
Inline documentation with comments
Reduced chance of file path errors


## 4. Creating Custom Configuration Files

Let's create a custom configuration file for a specific experiment:


In [7]:
# Create a custom configuration for a sensitivity analysis experiment
custom_config = SimulationConfig(
    simulation_name="Sensitivity_Analysis_2024",
    scenario="High_Growth_Scenario",
    intervention="Flood_Mitigation",
    start_year=2020,
    n_years=5,
    agent_housing_aggregation=20,  # Fewer agents for faster computation
    pop_growth_perc=0.02,          # Higher population growth
    perc_move=0.15,                # More agent movement
    simple_avoidance_perc=0.80,    # Lower flood avoidance
    price_increase_perc=0.08       # Higher price increases
)

print("=== Custom Configuration Created ===")
print(f"Simulation: {custom_config.simulation_name}")
print(f"Scenario: {custom_config.scenario}")
print(f"Intervention: {custom_config.intervention}")
print(f"Years: {custom_config.start_year} to {custom_config.start_year + custom_config.n_years}")
print(f"Agent aggregation: {custom_config.agent_housing_aggregation}")
print(f"Population growth: {custom_config.pop_growth_perc * 100}%")
print(f"Movement rate: {custom_config.perc_move * 100}%")
print(f"Flood avoidance: {custom_config.simple_avoidance_perc * 100}%")


=== Custom Configuration Created ===
Simulation: Sensitivity_Analysis_2024
Scenario: High_Growth_Scenario
Intervention: Flood_Mitigation
Years: 2020 to 2025
Agent aggregation: 20
Population growth: 2.0%
Movement rate: 15.0%
Flood avoidance: 80.0%


### 4.1 Saving Configuration to YAML

Now let's save our custom configuration to a YAML file:


In [8]:
# Save the custom configuration to a YAML file
output_config_path = "my_custom_config.yml"

custom_config.to_yaml(output_config_path)

print(f"=== Configuration Saved ===")
print(f"File saved to: {output_config_path}")
print()

# Read and display the saved YAML content
with open(output_config_path, 'r') as file:
    saved_yaml = file.read()

print("=== Saved YAML Content ===")
print(saved_yaml[:1000])  # Show first 1000 characters
if len(saved_yaml) > 1000:
    print("... (truncated)")

print(f"\nFile size: {len(saved_yaml)} characters")


=== Configuration Saved ===
File saved to: my_custom_config.yml

=== Saved YAML Content ===
agent_housing_aggregation: 20
bld_growth_perc: 0.01
block_group_sample_size: 10
budget_reduction_perc: 0.9
flood_file_mapping:
  GISJOIN: GISJOIN
  Shape_Area: Shape_Area
  fld_area: fld_area
  perc_fld_area: perc_fld_area
flood_filename: /Users/d3y010/repos/github/icom_abm/chance_c/data/example_input_data/block_group_percent_100yr_flood.csv
geo_file_mapping:
  ALAND: ALAND
  BLKGRPCE: BLKGRPCE
  COUNTYFP: COUNTYFP
  GEOID: GEOID
  GISJOIN: GISJOIN
  TRACTCE: TRACTCE
  geometry: geometry
geo_filename: /Users/d3y010/repos/github/icom_abm/chance_c/data/example_input_data/block_group_extract.shp
hedonic_file_mapping:
  GISJOIN: GISJOIN
  N_MeanAge: N_MeanAge
  N_MeanFullBathNumber: N_MeanFullBathNumber
  N_MeanNoOfStories: N_MeanNoOfStories
  N_MeanSqfeet: N_MeanSqfeet
  N_perc_area_flood: N_perc_area_flood
  residuals: residuals
hedonic_filename: /Users/d3y010/repos/github/icom_abm/chance_c/data/e

### 4.2 Verifying the Saved Configuration

Let's verify that we can load the saved configuration correctly:


In [9]:
# Load the saved configuration and verify it matches the original
loaded_config = SimulationConfig.from_yaml(output_config_path)

print("=== Configuration Verification ===")
print(f"Original simulation name: {custom_config.simulation_name}")
print(f"Loaded simulation name:   {loaded_config.simulation_name}")
print()

# Compare all parameters
all_match = True
mismatches = []

for param_name in custom_config.__dict__:
    original_value = getattr(custom_config, param_name)
    loaded_value = getattr(loaded_config, param_name)
    
    if original_value != loaded_value:
        all_match = False
        mismatches.append((param_name, original_value, loaded_value))

if all_match:
    print("Perfect match! All parameters loaded correctly.")
else:
    print(f"Found {len(mismatches)} mismatches:")
    for param_name, orig_val, loaded_val in mismatches:
        print(f"  {param_name}: {orig_val} → {loaded_val}")

print(f"\nTotal parameters verified: {len(custom_config.__dict__)}")


=== Configuration Verification ===
Original simulation name: Sensitivity_Analysis_2024
Loaded simulation name:   Sensitivity_Analysis_2024

Found 1 mismatches:
  simple_anova_coefficients: (-121428, 294707, 130553, 128990, 154887, -500000) → [-121428, 294707, 130553, 128990, 154887, -500000]

Total parameters verified: 40


## 5. Accessing Configuration from Model Class

The Model class provides seamless access to configuration parameters. Let's explore different ways to create and access configurations through the Model class.


In [10]:
# Method 1: Create Model with default configuration
print("=== Method 1: Default Configuration ===")
model_default = Model()
print(f"Model created with simulation: {model_default.config.simulation_name}")
print(f"Configuration type: {type(model_default.config).__name__}")
print(f"Start year: {model_default.config.start_year}")
print(f"Number of years: {model_default.config.n_years}")
print()

# Method 2: Create Model with custom parameters
print("=== Method 2: Custom Parameters ===")
model_custom = Model(
    simulation_name="Tutorial_Example",
    scenario="Learning_Session",
    start_year=2022,
    n_years=3,
    agent_housing_aggregation=15
)
print(f"Model created with simulation: {model_custom.config.simulation_name}")
print(f"Scenario: {model_custom.config.scenario}")
print(f"Start year: {model_custom.config.start_year}")
print(f"Agent aggregation: {model_custom.config.agent_housing_aggregation}")
print()

# Method 3: Create Model with existing configuration object
print("=== Method 3: Existing Configuration Object ===")
model_from_config = Model(config=custom_config)
print(f"Model created with simulation: {model_from_config.config.simulation_name}")
print(f"Scenario: {model_from_config.config.scenario}")
print(f"Population growth: {model_from_config.config.pop_growth_perc * 100}%")


=== Method 1: Default Configuration ===
Model created with simulation: ABM_Baltimore_example
Configuration type: SimulationConfig
Start year: 2018
Number of years: 2

=== Method 2: Custom Parameters ===
Model created with simulation: Tutorial_Example
Scenario: Learning_Session
Start year: 2022
Agent aggregation: 15

=== Method 3: Existing Configuration Object ===
Model created with simulation: Sensitivity_Analysis_2024
Scenario: High_Growth_Scenario
Population growth: 2.0%


### 5.1 Loading Model from YAML Configuration File

The most powerful method is to load a Model directly from a YAML configuration file:


In [11]:
# Method 4: Create Model from YAML file
print("=== Method 4: Load from YAML File ===")
model_from_yaml = Model(config_file_path=output_config_path)

print(f"Model loaded from: {output_config_path}")
print(f"Simulation name: {model_from_yaml.config.simulation_name}")
print(f"Scenario: {model_from_yaml.config.scenario}")
print(f"Intervention: {model_from_yaml.config.intervention}")
print(f"Years: {model_from_yaml.config.start_year}-{model_from_yaml.config.start_year + model_from_yaml.config.n_years}")
print()

# Accessing configuration parameters through the model
print("=== Accessing Configuration Parameters ===")
config = model_from_yaml.config

print("Simulation Parameters:")
print(f"  - Simulation name: {config.simulation_name}")
print(f"  - Scenario: {config.scenario}")
print(f"  - Start year: {config.start_year}")
print(f"  - Duration: {config.n_years} years")
print()

print("Agent Parameters:")
print(f"  - Housing aggregation: {config.agent_housing_aggregation}")
print(f"  - Household size: {config.household_size}")
print(f"  - Initial vacancy: {config.initial_vacancy * 100}%")
print()

print("Growth Parameters:")
print(f"  - Population growth: {config.pop_growth_perc * 100}% per year")
print(f"  - Income growth: {config.inc_growth_perc * 100}% per year")
print(f"  - Agent movement: {config.perc_move * 100}% per year")
print()

print("Housing Choice Parameters:")
print(f"  - Budget mode: {config.house_budget_mode}")
print(f"  - Choice mode: {config.house_choice_mode}")
print(f"  - Flood avoidance: {config.simple_avoidance_perc * 100}%")


=== Method 4: Load from YAML File ===
Model loaded from: my_custom_config.yml
Simulation name: Sensitivity_Analysis_2024
Scenario: High_Growth_Scenario
Intervention: Flood_Mitigation
Years: 2020-2025

=== Accessing Configuration Parameters ===
Simulation Parameters:
  - Simulation name: Sensitivity_Analysis_2024
  - Scenario: High_Growth_Scenario
  - Start year: 2020
  - Duration: 5 years

Agent Parameters:
  - Housing aggregation: 20
  - Household size: 2.7
  - Initial vacancy: 20.0%

Growth Parameters:
  - Population growth: 2.0% per year
  - Income growth: 5.0% per year
  - Agent movement: 15.0% per year

Housing Choice Parameters:
  - Budget mode: rhea
  - Choice mode: simple_avoidance_utility
  - Flood avoidance: 80.0%


## 6. Saving Configuration from Model Instance

After creating a model, you can save its configuration to a YAML file for future use or sharing:


In [12]:
# Create a model with specific parameters
experiment_model = Model(
    simulation_name="Sample_Study_2024",
    scenario="Sample",
    intervention="Sample",
    start_year=2025,
    n_years=10,
    agent_housing_aggregation=25,
    pop_growth_perc=0.015,
    perc_move=0.12,
    simple_avoidance_perc=0.85,
    price_increase_perc=0.06,
    budget_reduction_perc=0.85
)

print("=== Experiment Model Created ===")
print(f"Simulation: {experiment_model.config.simulation_name}")
print(f"Scenario: {experiment_model.config.scenario}")
print(f"Intervention: {experiment_model.config.intervention}")
print()

# Save the model's configuration
experiment_config_path = "sample_experiment_config.yml"
experiment_model.write_config(experiment_config_path)

print(f"=== Configuration Saved from Model ===")
print(f"Configuration saved to: {experiment_config_path}")
print(f"File exists: {os.path.exists(experiment_config_path)}")
print()

# Verify the saved configuration by loading it
verification_config = SimulationConfig.from_yaml(experiment_config_path)
print("=== Verification ===")
print(f"Original simulation name: {experiment_model.config.simulation_name}")
print(f"Loaded simulation name:   {verification_config.simulation_name}")
print(f"Match: {experiment_model.config.simulation_name == verification_config.simulation_name}")


2025-06-26 14:54:41,568 INFO: Config written to sample_experiment_config.yml


=== Experiment Model Created ===
Simulation: Sample_Study_2024
Scenario: Sample
Intervention: Sample

=== Configuration Saved from Model ===
Configuration saved to: sample_experiment_config.yml
File exists: True

=== Verification ===
Original simulation name: Sample_Study_2024
Loaded simulation name:   Sample_Study_2024
Match: True


### 6.1 Examining the Saved Configuration File


In [13]:
# Read and display the saved configuration file
with open(experiment_config_path, 'r') as file:
    saved_content = file.read()

print("=== Saved Configuration File Content ===")
print(f"File: {experiment_config_path}")
print(f"Size: {len(saved_content)} characters")
print()
print("Content preview (first 800 characters):")
print("-" * 50)
print(saved_content[:800])
if len(saved_content) > 800:
    print("... (truncated)")

# Show file structure
lines = saved_content.split('\n')
print(f"\nFile structure: {len(lines)} lines")

# Count parameter types
parameter_lines = [line for line in lines if ':' in line and not line.strip().startswith('#')]
print(f"Configuration parameters: {len(parameter_lines)}")

# Show some key parameters
print("\nKey parameters found:")
for line in lines[:20]:  # First 20 lines
    if ':' in line and not line.strip().startswith('#'):
        param_name = line.split(':')[0].strip()
        param_value = line.split(':', 1)[1].strip()
        print(f"  {param_name}: {param_value}")


=== Saved Configuration File Content ===
File: sample_experiment_config.yml
Size: 2215 characters

Content preview (first 800 characters):
--------------------------------------------------
agent_housing_aggregation: 25
bld_growth_perc: 0.01
block_group_sample_size: 10
budget_reduction_perc: 0.85
flood_file_mapping:
  GISJOIN: GISJOIN
  Shape_Area: Shape_Area
  fld_area: fld_area
  perc_fld_area: perc_fld_area
flood_filename: /Users/d3y010/repos/github/icom_abm/chance_c/data/example_input_data/block_group_percent_100yr_flood.csv
geo_file_mapping:
  ALAND: ALAND
  BLKGRPCE: BLKGRPCE
  COUNTYFP: COUNTYFP
  GEOID: GEOID
  GISJOIN: GISJOIN
  TRACTCE: TRACTCE
  geometry: geometry
geo_filename: /Users/d3y010/repos/github/icom_abm/chance_c/data/example_input_data/block_group_extract.shp
hedonic_file_mapping:
  GISJOIN: GISJOIN
  N_MeanAge: N_MeanAge
  N_MeanFullBathNumber: N_MeanFullBathNumber
  N_MeanNoOfStories: N_MeanNoOfStories
  N_MeanSqfeet: N_MeanSqfeet
  N_perc_area_flood: N
... (trun

## 7. Advanced Configuration Techniques

Let's explore some advanced techniques for working with configurations:


In [18]:
# 7.1 Configuration Validation and Field Mapping
print("=== Configuration Validation ===")

# Create a configuration with inline field mappings
config_with_mapping = SimulationConfig(
    simulation_name="Field_Mapping_Example",
    # Field mappings are now inline - no separate file needed!
    geo_file_mapping={
        "GISJOIN": "GISJOIN",
        "GEOID": "GEOID", 
        "COUNTYFP": "COUNTYFP",
        "TRACTCE": "TRACTCE",
        "BLKGRPCE": "BLKGRPCE",
        "ALAND": "ALAND",
        "geometry": "geometry"
    },
    pop_file_mapping={
        "GISJOIN": "GISJOIN",
        "AJWME001": "AJWME001"
    }
)

print(f"Configuration created with inline field mappings")
print(f"Geographic mappings: {len(config_with_mapping.geo_file_mapping)} fields")
print(f"Population mappings: {len(config_with_mapping.pop_file_mapping)} fields")
print()

# Validate field mapping
is_valid = config_with_mapping.validate_field_mapping()
print(f"Field mapping validation: {'Valid' if is_valid else 'Invalid'}")
print()

# Show the field mappings for different file types
print("=== Field Mapping Details ===")
file_types = [
    ('geo', config_with_mapping.geo_file_mapping),
    ('pop', config_with_mapping.pop_file_mapping),
    ('flood', config_with_mapping.flood_file_mapping),
    ('housing', config_with_mapping.housing_file_mapping),
    ('hedonic', config_with_mapping.hedonic_file_mapping)
]

for file_type, mapping in file_types:
    print(f"{file_type.upper()} file mappings:")
    if mapping:
        for field, column in list(mapping.items())[:3]:  # Show first 3
            print(f"  {field} → {column}")
        if len(mapping) > 3:
            print(f"  ... and {len(mapping) - 3} more fields")
    else:
        print("  No mappings defined")
    print()

print("=== Advantages of Inline Field Mappings ===")
print("All configuration in one file")
print("No risk of missing field mapping files")
print("Easy to customize column names")
print("Built-in validation")
print("Better version control")

# Optional: Show how to access the example field mapping file as package data
print()
print("=== Example Field Mapping File (Package Data) ===")
example_field_mapping_path = get_example_field_mapping_path()
print(f"Example field mapping file path: {example_field_mapping_path}")
print("This file now serves as a reference for migrating to inline mappings")


=== Configuration Validation ===
Configuration created with inline field mappings
Geographic mappings: 7 fields
Population mappings: 2 fields

Field mapping validation: Valid

=== Field Mapping Details ===
GEO file mappings:
  GISJOIN → GISJOIN
  GEOID → GEOID
  COUNTYFP → COUNTYFP
  ... and 4 more fields

POP file mappings:
  GISJOIN → GISJOIN
  AJWME001 → AJWME001

FLOOD file mappings:
  GISJOIN → GISJOIN
  Shape_Area → Shape_Area
  fld_area → fld_area
  ... and 1 more fields

HOUSING file mappings:
  GISJOIN → GISJOIN
  pop1990 → pop1990
  mhi1990 → mhi1990
  ... and 6 more fields

HEDONIC file mappings:
  GISJOIN → GISJOIN
  N_MeanSqfeet → N_MeanSqfeet
  N_MeanAge → N_MeanAge
  ... and 4 more fields

=== Advantages of Inline Field Mappings ===
All configuration in one file
No risk of missing field mapping files
Easy to customize column names
Built-in validation
Better version control

=== Example Field Mapping File (Package Data) ===
Example field mapping file path: /Users/d3y010/r

In [19]:
# 7.2 Configuration Comparison and Analysis
print("=== Configuration Comparison ===")

# Create two different configurations for comparison
config_baseline = SimulationConfig(
    simulation_name="Baseline_Scenario",
    scenario="Current_Conditions",
    pop_growth_perc=0.01,
    perc_move=0.10,
    simple_avoidance_perc=0.95
)

config_alternative = SimulationConfig(
    simulation_name="Alternative_Scenario", 
    scenario="Custom_Conditions",
    pop_growth_perc=0.005,  # Lower growth
    perc_move=0.20,         # More movement
    simple_avoidance_perc=0.75  # Less flood avoidance
)

# Compare configurations
def compare_configs(config1, config2, name1="Config 1", name2="Config 2"):
    """Compare two configurations and show differences."""
    differences = []
    
    for param_name in config1.__dict__:
        val1 = getattr(config1, param_name)
        val2 = getattr(config2, param_name)
        
        if val1 != val2:
            differences.append((param_name, val1, val2))
    
    print(f"Comparing {name1} vs {name2}:")
    print(f"Found {len(differences)} differences:")
    print()
    
    for param_name, val1, val2 in differences:
        print(f"  {param_name}:")
        print(f"    {name1}: {val1}")
        print(f"    {name2}: {val2}")
        print()
    
    return differences

differences = compare_configs(
    config_baseline, 
    config_alternative,
    "Baseline",
    "Alternative"
)


=== Configuration Comparison ===
Comparing Baseline vs Alternative:
Found 5 differences:

  simulation_name:
    Baseline: Baseline_Scenario
    Alternative: Alternative_Scenario

  scenario:
    Baseline: Current_Conditions
    Alternative: Climate_Change_Impact

  pop_growth_perc:
    Baseline: 0.01
    Alternative: 0.005

  perc_move:
    Baseline: 0.1
    Alternative: 0.2

  simple_avoidance_perc:
    Baseline: 0.95
    Alternative: 0.75



In [20]:
# 7.3 Batch Configuration Creation for Sensitivity Analysis
print("=== Batch Configuration Creation ===")

# Create multiple configurations for sensitivity analysis
base_params = {
    "simulation_name": "Sensitivity_Analysis",
    "scenario": "Parameter_Sweep",
    "start_year": 2020,
    "n_years": 3
}

# Parameters to vary
pop_growth_values = [0.005, 0.01, 0.015, 0.02]
move_percentages = [0.05, 0.10, 0.15, 0.20]

print("Creating sensitivity analysis configurations...")
configs = []

for i, pop_growth in enumerate(pop_growth_values):
    for j, move_perc in enumerate(move_percentages):
        config = SimulationConfig(
            simulation_name=f"Sensitivity_PopGrowth_{pop_growth:.3f}_Move_{move_perc:.2f}",
            scenario="Parameter_Sweep",
            start_year=2020,
            n_years=3,
            pop_growth_perc=pop_growth,
            perc_move=move_perc
        )
        configs.append(config)
        
        # Save each configuration
        filename = f"sensitivity_config_{i}_{j}.yml"
        config.to_yaml(filename)

print(f"Created {len(configs)} configurations:")
for i, config in enumerate(configs):
    print(f"  {i+1:2d}. {config.simulation_name}")
    print(f"      Pop growth: {config.pop_growth_perc:.3f}, Move: {config.perc_move:.2f}")

print(f"\nAll configurations saved as YAML files.")


=== Batch Configuration Creation ===
Creating sensitivity analysis configurations...
Created 16 configurations:
   1. Sensitivity_PopGrowth_0.005_Move_0.05
      Pop growth: 0.005, Move: 0.05
   2. Sensitivity_PopGrowth_0.005_Move_0.10
      Pop growth: 0.005, Move: 0.10
   3. Sensitivity_PopGrowth_0.005_Move_0.15
      Pop growth: 0.005, Move: 0.15
   4. Sensitivity_PopGrowth_0.005_Move_0.20
      Pop growth: 0.005, Move: 0.20
   5. Sensitivity_PopGrowth_0.010_Move_0.05
      Pop growth: 0.010, Move: 0.05
   6. Sensitivity_PopGrowth_0.010_Move_0.10
      Pop growth: 0.010, Move: 0.10
   7. Sensitivity_PopGrowth_0.010_Move_0.15
      Pop growth: 0.010, Move: 0.15
   8. Sensitivity_PopGrowth_0.010_Move_0.20
      Pop growth: 0.010, Move: 0.20
   9. Sensitivity_PopGrowth_0.015_Move_0.05
      Pop growth: 0.015, Move: 0.05
  10. Sensitivity_PopGrowth_0.015_Move_0.10
      Pop growth: 0.015, Move: 0.10
  11. Sensitivity_PopGrowth_0.015_Move_0.15
      Pop growth: 0.015, Move: 0.15
  12. Se

## 8. Best Practices and Tips

Here are some best practices for working with CHANCE-C configurations:


### 8.3 Inline Field Mapping Best Practices

With the new inline field mapping system, here are additional best practices:

1. **Keep Mappings Consistent**: Use the same identifier field (e.g., GISJOIN) across all file types
2. **Document Custom Mappings**: Add comments in YAML files explaining non-standard column names
3. **Validate Before Running**: Always test your field mappings with small datasets first
4. **Use Descriptive Column Names**: Choose clear, meaningful column names in your data files
5. **Version Control Mappings**: Track field mapping changes in Git alongside your data
6. **Test with Sample Data**: Verify mappings work with your actual data files
7. **Backup Working Configurations**: Save configurations that successfully load your data


In [22]:
# Demonstrate inline field mapping validation and best practices
print("=== Inline Field Mapping Validation ===")

# Example: Create a configuration with custom field mappings
validation_config = SimulationConfig(
    simulation_name="Field_Mapping_Validation_Example",
    scenario="Custom_Data_Integration",
    
    # Custom mappings for your data files
    geo_file_mapping={
        "GISJOIN": "block_group_id",      # Your shapefile uses this column
        "GEOID": "geoid_2010",            # Your shapefile uses this column
        "COUNTYFP": "county_fips",        # Your shapefile uses this column
        "TRACTCE": "tract_code",          # Your shapefile uses this column
        "BLKGRPCE": "bg_code",            # Your shapefile uses this column
        "ALAND": "land_area_sqm",         # Your shapefile uses this column
        "geometry": "geometry"            # Standard geometry column
    },
    
    pop_file_mapping={
        "GISJOIN": "block_group_id",      # Match the geo file
        "AJWME001": "total_pop_2018"      # Your population file uses this column
    },
    
    flood_file_mapping={
        "GISJOIN": "block_group_id",      # Match the geo file
        "Shape_Area": "total_area",       # Your flood file uses this column
        "fld_area": "flood_area_sqm",     # Your flood file uses this column
        "perc_fld_area": "flood_percent"  # Your flood file uses this column
    }
)

print("Configuration created with custom field mappings")
print()

# Show the field mappings
print("=== Field Mapping Summary ===")
mapping_types = [
    ("Geographic", validation_config.geo_file_mapping),
    ("Population", validation_config.pop_file_mapping), 
    ("Flood", validation_config.flood_file_mapping),
    ("Housing", validation_config.housing_file_mapping),
    ("Hedonic", validation_config.hedonic_file_mapping)
]

for mapping_name, mapping_dict in mapping_types:
    print(f"{mapping_name} file mappings ({len(mapping_dict)} fields):")
    if mapping_dict:
        for required_field, your_column in list(mapping_dict.items())[:3]:
            print(f"  {required_field:12} → {your_column}")
        if len(mapping_dict) > 3:
            print(f"  ... and {len(mapping_dict) - 3} more fields")
    print()

# Validate the field mappings
print("=== Field Mapping Validation ===")
try:
    is_valid = validation_config.validate_field_mapping()
    print(f"Field mapping validation: {'Valid' if is_valid else 'Invalid'}")
except Exception as e:
    print(f"Validation error: {e}")

print()
print("=== Key Advantages of Inline Field Mappings ===")
print("All configuration in one file - no separate mapping files")
print("Easy to customize for different data sources")
print("Built-in validation ensures mappings are complete")
print("Version control tracks all changes together")
print("Reduced risk of missing or mismatched files")
print("Inline documentation with comments")
print("Simplified deployment and sharing")


=== Inline Field Mapping Validation ===
Configuration created with custom field mappings

=== Field Mapping Summary ===
Geographic file mappings (7 fields):
  GISJOIN      → block_group_id
  GEOID        → geoid_2010
  COUNTYFP     → county_fips
  ... and 4 more fields

Population file mappings (2 fields):
  GISJOIN      → block_group_id
  AJWME001     → total_pop_2018

Flood file mappings (4 fields):
  GISJOIN      → block_group_id
  Shape_Area   → total_area
  fld_area     → flood_area_sqm
  ... and 1 more fields

Housing file mappings (9 fields):
  GISJOIN      → GISJOIN
  pop1990      → pop1990
  mhi1990      → mhi1990
  ... and 6 more fields

Hedonic file mappings (7 fields):
  GISJOIN      → GISJOIN
  N_MeanSqfeet → N_MeanSqfeet
  N_MeanAge    → N_MeanAge
  ... and 4 more fields

=== Field Mapping Validation ===
Field mapping validation: Valid

=== Key Advantages of Inline Field Mappings ===
All configuration in one file - no separate mapping files
Easy to customize for different

### 8.1 Configuration Management Best Practices

1. **Use Descriptive Names**: Choose clear, descriptive names for your simulations
2. **Version Control**: Store configuration files in version control (Git)
3. **Document Changes**: Use meaningful scenario and intervention names
4. **Validate Configurations**: Always validate field mappings before running
5. **Organize Files**: Keep configurations organized in directories by project
6. **Backup Important Configs**: Save successful experiment configurations
7. **Use Comments**: Add comments to YAML files to explain parameter choices

### 8.2 Common Patterns

#### Pattern 1: Experiment Series
```python
# Create a base configuration
base_config = SimulationConfig(simulation_name="Base_Experiment")

# Modify for different scenarios
scenario_a = SimulationConfig(**base_config.__dict__)
scenario_a.simulation_name = "Scenario_A"
scenario_a.pop_growth_perc = 0.015

scenario_b = SimulationConfig(**base_config.__dict__)
scenario_b.simulation_name = "Scenario_B" 
scenario_b.pop_growth_perc = 0.005
```

#### Pattern 2: Configuration Templates
```python
# Create template configurations for different study types
example_template = {
    "scenario": "Example",
    "n_years": 10,
    "pop_growth_perc": 0.008
}

policy_template = {
    "scenario": "Policy_Analysis", 
    "n_years": 5,
    "intervention": "Zoning_Change"
}
```


In [23]:
# 8.3 Configuration Summary and Cleanup
print("=== Configuration Tutorial Summary ===")
print()

# List all configuration files created in this tutorial
created_files = [
    "my_custom_config.yml",
    "custom_experiment_config.yml"
]

# Add sensitivity analysis files
for i in range(4):
    for j in range(4):
        created_files.append(f"sensitivity_config_{i}_{j}.yml")

print(f"Files created in this tutorial: {len(created_files)}")
for i, filename in enumerate(created_files, 1):
    if os.path.exists(filename):
        file_size = os.path.getsize(filename)
        print(f"  {i:2d}. {filename} ({file_size:,} bytes)")
    else:
        print(f"  {i:2d}. {filename} (not found)")

print()
print("=== Key Takeaways ===")
print("1. SimulationConfig class manages all simulation parameters")
print("2. YAML files provide human-readable configuration storage")
print("3. Model class seamlessly integrates with configurations")
print("4. Configurations can be saved from Model instances")
print("5. Inline field mappings eliminate separate mapping files")
print("6. Custom field mappings support different data column names")
print("7. Batch configuration creation supports sensitivity analysis")
print("8. Configuration comparison helps analyze experiment differences")
print("9. Built-in validation ensures field mappings are complete")
print("10. All configuration settings consolidated in single files")
print()

# Cleanup option (commented out to preserve files for inspection)
# print("=== Cleanup ===")
# print("Uncomment the following code to clean up created files:")
# print()
# print("# for filename in created_files:")
# print("#     if os.path.exists(filename):")
# print("#         os.remove(filename)")
# print("#         print(f'Removed {filename}')")

print("Configuration tutorial completed successfully!")


=== Configuration Tutorial Summary ===

Files created in this tutorial: 18
   1. my_custom_config.yml (2,243 bytes)
   2. climate_experiment_config.yml (not found)
   3. sensitivity_config_0_0.yml (2,244 bytes)
   4. sensitivity_config_0_1.yml (2,243 bytes)
   5. sensitivity_config_0_2.yml (2,244 bytes)
   6. sensitivity_config_0_3.yml (2,243 bytes)
   7. sensitivity_config_1_0.yml (2,243 bytes)
   8. sensitivity_config_1_1.yml (2,242 bytes)
   9. sensitivity_config_1_2.yml (2,243 bytes)
  10. sensitivity_config_1_3.yml (2,242 bytes)
  11. sensitivity_config_2_0.yml (2,244 bytes)
  12. sensitivity_config_2_1.yml (2,243 bytes)
  13. sensitivity_config_2_2.yml (2,244 bytes)
  14. sensitivity_config_2_3.yml (2,243 bytes)
  15. sensitivity_config_3_0.yml (2,243 bytes)
  16. sensitivity_config_3_1.yml (2,242 bytes)
  17. sensitivity_config_3_2.yml (2,243 bytes)
  18. sensitivity_config_3_3.yml (2,242 bytes)

=== Key Takeaways ===
1. SimulationConfig class manages all simulation parameters
2

## 9. Next Steps and Resources

### 9.1 Additional Resources

- **Quickstarter Notebook**: `notebooks/quickstarter.ipynb` - Get started with CHANCE-C
- **Example Configuration**: `chance_c/data/example_config.yml` - Complete configuration template with inline field mappings
- **Field Mapping Guide**: `chance_c/data/FIELD_MAPPING_README.md` - Inline field mapping documentation
- **CLI Documentation**: `CLI_README.md` - Command-line interface usage

### 9.2 Recommended Workflow

1. **Start with defaults**: Use `Model()` for initial exploration
2. **Customize gradually**: Modify parameters as needed for your study
3. **Configure field mappings**: Set up inline field mappings for your data files
4. **Validate configurations**: Test field mappings with your actual data
5. **Save configurations**: Always save successful experiment configurations
6. **Version control**: Track configuration changes in Git
7. **Document experiments**: Use descriptive names and inline comments

### 9.3 Advanced Topics

- **Parallel simulations**: Run multiple configurations simultaneously
- **Parameter optimization**: Use optimization libraries with CHANCE-C configurations
- **Reproducible research**: Combine configurations with random seeds
- **Cloud deployment**: Scale simulations using cloud computing resources

This completes the comprehensive CHANCE-C configuration tutorial. You now have the knowledge to effectively manage configurations with the new inline field mapping system for your agent-based modeling research!

### 🎉 What You've Learned

- **Configuration Management**: Create, modify, and save CHANCE-C configurations
- **Inline Field Mappings**: Set up custom field mappings directly in configuration files
- **Data Integration**: Connect your own data files using flexible column mappings
- **Validation**: Ensure your configurations are complete and correct
- **Best Practices**: Follow proven patterns for reproducible research
- **Advanced Techniques**: Create batch configurations for sensitivity analysis

You're now ready to use CHANCE-C with your own data and research questions!
